# LazyAd — Spike S1: Colab Diagnostic & Hardware Profiling

**Objective:** Query and log the active Google Colab runtime environment:
- GPU model, VRAM capacity, and compute capability
- CPU specs and total available system RAM
- Ephemeral disk space in `/content`
- Python, PyTorch, and CUDA versions
- Local I/O speed benchmark

**Pass Criteria:** Script executes cleanly, confirms GPU availability, and prints hardware metrics required to configure subsequent stages (FLUX.1 schnell and TripoSR).

### Step 1: Check GPU & Hardware via `nvidia-smi`

In [ ]:
!nvidia-smi

### Step 2: PyTorch CUDA & Environment Diagnostics

In [ ]:
import sys
import os
import time
import shutil
import psutil
import torch

print("=" * 60)
print("LAZYAD — SPIKE S1: RUNTIME DIAGNOSTIC REPORT")
print("=" * 60)

# 1. Python & PyTorch Versions
print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version     : {torch.version.cuda}")
    print(f"Device Name      : {torch.cuda.get_device_name(0)}")
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total VRAM       : {total_vram_gb:.2f} GB")
else:
    print("\n[WARNING] No GPU detected! Please go to: Runtime -> Change runtime type -> Select T4 GPU or A100 GPU.")

# 2. System RAM
ram = psutil.virtual_memory()
total_ram_gb = ram.total / (1024 ** 3)
avail_ram_gb = ram.available / (1024 ** 3)
print(f"System RAM       : {total_ram_gb:.2f} GB total ({avail_ram_gb:.2f} GB available)")

# 3. Disk Space in /content
disk = shutil.disk_usage("/content")
free_disk_gb = disk.free / (1024 ** 3)
total_disk_gb = disk.total / (1024 ** 3)
print(f"Disk Space       : {total_disk_gb:.2f} GB total ({free_disk_gb:.2f} GB free)")
print("=" * 60)


### Step 3: Local Disk I/O Benchmark (100 MB write/read)

In [ ]:
# Benchmark ephemeral disk I/O performance
test_file = "/content/io_test.bin"
size_mb = 100
data = os.urandom(size_mb * 1024 * 1024)

# Write test
start_time = time.time()
with open(test_file, "wb") as f:
    f.write(data)
write_time = time.time() - start_time
write_speed = size_mb / write_time

# Read test
start_time = time.time()
with open(test_file, "rb") as f:
    _ = f.read()
read_time = time.time() - start_time
read_speed = size_mb / read_time

# Clean up
if os.path.exists(test_file):
    os.remove(test_file)

print(f"Disk Write Speed : {write_speed:.2f} MB/s ({size_mb} MB in {write_time:.2f}s)")
print(f"Disk Read Speed  : {read_speed:.2f} MB/s ({size_mb} MB in {read_time:.2f}s)")
print("=" * 60)


### Step 4: Hardware Compatibility Assessment for LazyAd

In [ ]:
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    
    print("\n--- LAZYAD MODEL PROFILE READINESS ---")
    
    # 3D Model Readiness
    if vram >= 6.0:
        print("✅ Baseline 3D (TripoSR)      : READY (~6 GB VRAM required)")
    else:
        print("❌ Baseline 3D (TripoSR)      : NOT ENOUGH VRAM")
        
    if vram >= 16.0:
        print("✅ Advanced 3D (TRELLIS v1)    : CANDIDATE (>=16 GB VRAM detected)")
    else:
        print("ℹ️ Advanced 3D (TRELLIS v1)    : DEFERRED (Requires >=16 GB VRAM)")
        
    # Image Gen Readiness
    if vram >= 14.0:
        print("✅ Image Gen (FLUX.1 schnell) : READY (Will run with CPU offload/FP8 on T4 or direct on A100)")
    else:
        print("⚠️ Image Gen (FLUX.1 schnell) : TIGHT VRAM - Recommend SDXL-Lightning fallback")
        
    print("\n🎉 SPIKE S1 DIAGNOSTIC COMPLETE: Please copy and share these results back with Antigravity!")
else:
    print("\n❌ GPU not found. Switch to a GPU runtime in Colab and re-run.")
